# Jurimetria DataJud no Google Colab

Este notebook sobe **o app completo em Streamlit** dentro do Colab, usando a versao atual do repositorio.


## Como usar

1. Execute a celula de clone/sincronizacao do repositorio.
2. Execute a instalacao das dependencias.
3. Informe sua chave da API DataJud.
4. Inicie o Streamlit.
5. Abra o link publico gerado pelo Cloudflare Tunnel.

Depois disso, a interface do app fica igual a da versao web, mas rodando a partir do Colab.


In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/lucaslmfbib/jurimetria-datajud.git"
REPO_BRANCH = "main"
REPO_DIR = Path("/content/jurimetria-datajud")

if not REPO_DIR.exists():
    !git clone --depth 1 --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}
else:
    %cd {REPO_DIR}
    !git fetch origin {REPO_BRANCH}
    !git checkout {REPO_BRANCH}
    !git pull --ff-only origin {REPO_BRANCH}

%cd {REPO_DIR}
print(f"Repositorio pronto em: {REPO_DIR}")


In [ ]:
%cd /content/jurimetria-datajud
!pip -q install -r requirements.txt
!wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared
print("Dependencias instaladas e tunel pronto.")


In [ ]:
import getpass
import os

api_key = os.environ.get("DATAJUD_API_KEY", "").strip()
if not api_key:
    api_key = getpass.getpass("Cole sua chave DataJud (APIKey ...): ").strip()

if api_key and not api_key.startswith(("APIKey ", "Bearer ")):
    api_key = f"APIKey {api_key}"

os.environ["DATAJUD_API_KEY"] = api_key
print("Chave carregada no ambiente." if api_key else "Chave ausente.")


In [ ]:
import subprocess
import time
from pathlib import Path

REPO_DIR = Path("/content/jurimetria-datajud")
PORT = 8501

def stop_process(proc_name: str) -> None:
    proc = globals().get(proc_name)
    if proc and proc.poll() is None:
        proc.terminate()
        try:
            proc.wait(timeout=5)
        except subprocess.TimeoutExpired:
            proc.kill()

stop_process("streamlit_proc")

streamlit_proc = subprocess.Popen(
    [
        "streamlit",
        "run",
        "streamlit_app.py",
        "--server.port",
        str(PORT),
        "--server.headless",
        "true",
        "--browser.gatherUsageStats",
        "false",
        "--server.fileWatcherType",
        "none",
    ],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(8)
if streamlit_proc.poll() is None:
    print(f"Streamlit iniciado em http://127.0.0.1:{PORT}")
else:
    print("O Streamlit encerrou logo apos iniciar. Rode a celula de logs para inspecionar.")


In [ ]:
import re
import subprocess
import time
from IPython.display import Markdown, display
from pathlib import Path

REPO_DIR = Path("/content/jurimetria-datajud")
PORT = 8501
TUNNEL_BIN = REPO_DIR / "cloudflared"

stop_process("tunnel_proc")

tunnel_proc = subprocess.Popen(
    [str(TUNNEL_BIN), "tunnel", "--url", f"http://127.0.0.1:{PORT}"],
    cwd=REPO_DIR,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
pattern = re.compile(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com")
start = time.time()
while time.time() - start < 40:
    line = tunnel_proc.stdout.readline()
    if not line:
        continue
    line = line.strip()
    print(line)
    match = pattern.search(line)
    if match:
        public_url = match.group(0)
        break

if public_url:
    display(Markdown(f"### Abra o app aqui: [{public_url}]({public_url})"))
else:
    print("Nao consegui capturar a URL publica. Rode esta celula novamente.")


In [ ]:
def mostrar_logs(proc_name: str = "streamlit_proc", max_linhas: int = 80) -> None:
    proc = globals().get(proc_name)
    if not proc:
        print("Processo nao iniciado.")
        return
    if proc.stdout is None:
        print("Sem saida disponivel.")
        return
    lidas = 0
    while lidas < max_linhas:
        line = proc.stdout.readline()
        if not line:
            break
        print(line.rstrip())
        lidas += 1

mostrar_logs("streamlit_proc", 80)


In [ ]:
stop_process("tunnel_proc")
stop_process("streamlit_proc")
print("Processos encerrados.")
